# 14. Factor Analysis (FA)

You never directly measure "how electrochemically active a sample is" — you
only measure things like corrosion current, mass loss, and pitting depth.
But if an underlying property like "electrochemical activity" genuinely
exists, all three of those measurements should move together, driven by
that one hidden cause. Factor Analysis works backwards from the pattern of
correlations between observed variables to recover these hidden ("latent")
common causes:

$$\mathbf{x} = \boldsymbol{\Lambda} \mathbf{f} + \boldsymbol{\varepsilon}$$

where $\boldsymbol{\Lambda}$ (factor loadings) says how strongly each
observed variable responds to each hidden factor $\mathbf{f}$, and
$\boldsymbol{\varepsilon}$ is whatever is left over in each variable that
isn't explained by the shared factors — its own private noise. See Section
5 of the [theory page](theory.md) for the full intuition and how this
differs from PCA.

**Key difference from PCA**: PCA maximises explained variance; FA explicitly
models the **common variance** shared among variables and ignores unique variance.

**Topics**
1. FA vs PCA: conceptual differences
2. Determining the number of factors (scree plot, parallel analysis)
3. Factor loadings and communalities
4. Rotation: varimax and promax
5. Interpreting factors
6. Case study: corrosion test battery data

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
from sklearn.preprocessing import StandardScaler

# factor_analyzer 0.5.1 (see requirements.txt) internally passes a scikit-learn
# validation kwarg, 'force_all_finite', that sklearn>=1.6 renamed to
# 'ensure_all_finite' -- a harmless library-version mismatch, not a data
# problem. Left unsilenced, the parallel-analysis loop below (Section 14.3)
# calls FactorAnalyzer.fit() 1,000 times and would print this same notice
# 1,000 times, so it is switched off here, specifically, rather than hiding
# warnings in general.
warnings.filterwarnings('ignore', message='.*force_all_finite.*', category=FutureWarning)

sns.set_theme(style='ticks', palette='colorblind')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
rng = np.random.default_rng(37)

## 14.1 Generate Corrosion Test Dataset

The dataset contains 10 electrochemical and physical measurements from 80
corrosion tests on a zinc-coated steel in varying NaCl concentration, pH,
temperature, and coating thickness:

| Variable | Description |
|---|---|
| `E_corr` | Corrosion potential (mV vs SCE) |
| `i_corr` | Corrosion current density (μA/cm²) |
| `R_p` | Polarisation resistance (kΩ·cm²) |
| `EIS_R1` | High-frequency EIS resistance (Ω·cm²) |
| `EIS_R2` | Low-frequency EIS resistance (kΩ·cm²) |
| `mass_loss` | Mass loss after 72 h (mg/cm²) |
| `pit_depth` | Max pitting depth (μm) |
| `coating_t` | Coating thickness (μm) |
| `NaCl_pct` | NaCl concentration (%) |
| `temp_C` | Test temperature (°C) |

The dataset is designed so that three underlying factors are present:
- **Factor 1** — *Electrochemical activity* (i_corr, mass_loss, pit_depth)
- **Factor 2** — *Barrier quality* (R_p, EIS_R2, coating_t)
- **Factor 3** — *Test aggressiveness* (NaCl_pct, temp_C, EIS_R1)

In [ ]:
n = 80

# Latent factors
f_activity    = rng.normal(0, 1, n)  # electrochemical activity
f_barrier     = rng.normal(0, 1, n)  # coating / barrier quality
f_aggressive  = rng.normal(0, 1, n)  # test aggressiveness

noise = lambda s: rng.normal(0, s, n)

E_corr  = -500 - 30*f_activity + 10*f_barrier   + noise(12)
i_corr  =   20 + 8*f_activity  - 2*f_barrier    + noise(2)
R_p     =  100 - 20*f_activity + 35*f_barrier   + noise(8)
EIS_R1  =   15 - 2*f_activity  +  3*f_aggressive + noise(2)
EIS_R2  =  200 - 30*f_activity + 60*f_barrier   + noise(15)
mass_loss = 3  +  1.5*f_activity - 0.5*f_barrier + noise(0.3)
pit_depth = 80 + 25*f_activity  -  8*f_barrier   + noise(10)
coating_t = 15 -  2*f_activity  +  5*f_barrier   + noise(1)
NaCl_pct  =  2 +  0.8*f_aggressive               + noise(0.2)
temp_C    = 35 +  6*f_aggressive                 + noise(2)

df_corr = pd.DataFrame({
    'E_corr':   E_corr,
    'i_corr':   i_corr,
    'R_p':      R_p,
    'EIS_R1':   EIS_R1,
    'EIS_R2':   EIS_R2,
    'mass_loss': mass_loss,
    'pit_depth': pit_depth,
    'coating_t': coating_t,
    'NaCl_pct':  NaCl_pct,
    'temp_C':    temp_C,
})
print(df_corr.describe().round(2))

## 14.2 Suitability Checks

FA only makes sense if the variables actually share enough common variance
to begin with — if all ten measurements were essentially independent of
each other, there would be no hidden common factor to recover, and FA would
just be fitting noise. Two checks confirm there's enough shared structure
before proceeding:

- **Bartlett's sphericity test**: tests $H_0$ = "the correlation matrix is
  essentially an identity matrix" (i.e. all variables are uncorrelated).
  You *want* to reject this — a significant result (small p-value) confirms
  the variables are correlated enough for FA to be meaningful.
- **Kaiser-Meyer-Olkin (KMO)** measure of sampling adequacy: roughly, "how
  much of the correlation between variables is likely to reflect real
  shared factors, versus noise?" > 0.6 acceptable, > 0.8 good.

In [ ]:
chi_sq, p_val = calculate_bartlett_sphericity(df_corr)
kmo_per_var, kmo_overall = calculate_kmo(df_corr)

print(f'Bartlett sphericity: χ² = {chi_sq:.2f}, p = {p_val:.4f}')
print(f'KMO overall: {kmo_overall:.3f}')
print(f'KMO per variable:\n{pd.Series(kmo_per_var, index=df_corr.columns).round(3).to_string()}')

:::{admonition} Take-home message
:class: tip

- Both checks pass comfortably: Bartlett's p≈0 confirms the ten variables are far from uncorrelated (there is real shared structure to factor), and KMO=0.852 clears even the stricter "good" threshold of 0.8, not just the 0.6 minimum — this dataset is well suited to FA.
- Every per-variable KMO sits between 0.66 and 0.94, with `NaCl_pct` (0.661) and `temp_C` (0.665) the weakest — both still acceptable, but notably lower than the electrochemical measurements. That is a hint, confirmed in Section 14.4, that these two share strong information with each other but somewhat less with the rest of the variable set.
:::

## 14.3 Number of Factors — Scree Plot and Parallel Analysis

In [ ]:
# FA without rotation to see eigenvalues
fa_full = FactorAnalyzer(n_factors=10, rotation=None)
fa_full.fit(df_corr)
ev, v = fa_full.get_eigenvalues()

# Parallel analysis: eigenvalues from random data (95th percentile)
n_sims = 1000
rand_ev = np.zeros((n_sims, len(ev)))
for s in range(n_sims):
    rand_data = rng.normal(0, 1, df_corr.shape)
    fa_rand = FactorAnalyzer(n_factors=len(ev), rotation=None)
    fa_rand.fit(pd.DataFrame(rand_data, columns=df_corr.columns))
    rand_ev[s] = fa_rand.get_eigenvalues()[0]
parallel_thresh = np.percentile(rand_ev, 95, axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(ev)+1), ev, 'bo-', lw=2, ms=8, label='Observed eigenvalues')
ax.plot(range(1, len(ev)+1), parallel_thresh, 'r--s', lw=1.5, ms=7,
        label='Parallel analysis (95%)')
ax.axhline(1, color='gray', ls=':', lw=1, label='Kaiser criterion (λ=1)')
ax.set_xlabel('Factor number')
ax.set_ylabel('Eigenvalue')
ax.set_title('Scree Plot with Parallel Analysis')
ax.legend(fontsize=9)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

print('\nEigenvalues:', ev.round(3))

:::{admonition} Take-home message
:class: tip

- The eigenvalues drop sharply after the third: 5.76, 2.39, 1.25, then 0.17 — a clean cliff, not a gradual taper. All three of the first eigenvalues clear both the Kaiser criterion (>1) and the parallel-analysis threshold, while the fourth (0.17) is nowhere close to either — an unusually unambiguous case for "how many factors," which makes sense given this dataset was deliberately built from exactly three latent causes (Section 14.1).
:::

### Interpreting the Scree Plot and Parallel Analysis

- **Parallel analysis** is the gold standard for choosing the number of factors. Keep factors whose observed eigenvalue (blue circles) lies **above** the simulated random-data threshold (red dashes). In this corrosion dataset the threshold clearly supports retaining three factors.
- **Kaiser criterion (λ = 1)** tends to over-factor. Use it only as a secondary guide.
- **Eigenvalue elbow** — the point where the curve 'bends' also suggests the number of meaningful factors. Factors beyond the elbow explain little more than noise.

## 14.4 Factor Analysis with Varimax Rotation

In [ ]:
n_factors = 3
fa = FactorAnalyzer(n_factors=n_factors, rotation='varimax')
fa.fit(df_corr)

loadings = fa.loadings_
communalities = fa.get_communalities()
factor_names = [f'F{i+1}' for i in range(n_factors)]

load_df = pd.DataFrame(loadings, index=df_corr.columns, columns=factor_names)
load_df['Communality'] = communalities
print('Factor Loadings (varimax rotated):')
print(load_df.round(3).to_string())

:::{admonition} Take-home message
:class: tip

- The loadings recover the three intended factors almost exactly as designed: F1 loads strongly on `i_corr` (+0.92), `mass_loss` (+0.91), and `pit_depth` (+0.86) with `E_corr` swinging opposite (−0.87) — all the *electrochemical activity* variables, and nothing else. F2 loads on `coating_t` (+0.94), `EIS_R2` (+0.90), and `R_p` (+0.88) — *barrier quality*. F3 loads on `NaCl_pct` (+0.98) and `temp_C` (+0.96) — *test aggressiveness*; `EIS_R1`, also designed to belong to F3, is the one variable that doesn't come through as cleanly (see below).
- Every variable's loading on its "own" factor is well above 0.85 **except `EIS_R1`**, whose loading on F3 is only +0.77 — its cross-loadings still stay small (F1=−0.48, F2=+0.05), so it isn't being pulled toward a *different* factor, it's just a noisier read of test aggressiveness than `NaCl_pct` or `temp_C`. That traces directly back to how it was generated (Section 14.1): `EIS_R1 = 15 − 2·f_activity + 3·f_aggressive + noise(2)` genuinely mixes in a real `f_activity` contribution, unlike `NaCl_pct` and `temp_C`, which depend on `f_aggressive` alone — varimax can sharpen a loading, but it can't remove a dependency that was actually built into the data.
- Communalities are all ≥ 0.82 (`EIS_R1` lowest at 0.825), meaning the three common factors explain the large majority of every variable's variance — very little is left over as variable-specific noise, again consistent with how cleanly this teaching dataset was constructed.
:::

In [ ]:
# Loading heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(load_df[factor_names], annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax)
ax.set_title('Varimax Rotated Factor Loadings')
ax.set_xlabel('Factor')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.show()

### Interpreting the Factor Loading Heatmap

The heatmap shows the correlation between each *observed variable* (rows) and each *latent factor* (columns) after varimax rotation:

- **Large positive loading (deep red, ≈ +1)** — the variable increases strongly with this factor.
- **Large negative loading (deep blue, ≈ −1)** — the variable decreases as this factor increases.
- **Near-zero loading (white)** — the variable is essentially unrelated to this factor; it is explained by another factor or by unique variance.
- **Varimax rotation** rotates the loading matrix so that each variable loads high on as few factors as possible. This makes factors easier to interpret as distinct latent constructs.
- **Communality** (rightmost column) is the fraction of each variable's variance explained by *all* retained factors combined. Low communality (< 0.40) suggests a variable is mostly unique variance and carries little shared information.
- **Naming factors** — assign a meaningful label to each factor based on the cluster of variables with high loadings. Here: F1 ≈ electrochemical activity, F2 ≈ barrier quality, F3 ≈ test aggressiveness.

## 14.5 Factor Scores and Interpretation

In [ ]:
factor_scores = fa.transform(df_corr)
score_df = pd.DataFrame(factor_scores, columns=['Electrochemical\nactivity',
                                                  'Barrier\nquality',
                                                  'Test\naggressiveness'])

# Pairplot of factor scores
g = sns.pairplot(score_df, diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20})
g.fig.suptitle('Pairplot of Factor Scores', y=1.02)
plt.show()

# Variance explained
var_df = pd.DataFrame(fa.get_factor_variance(),
                      index=['SS Loadings', 'Proportion Variance', 'Cumulative Variance'],
                      columns=factor_names)
print('\nVariance explained:')
print(var_df.round(3))

:::{admonition} Take-home message
:class: tip

- The three factors split the variance almost evenly (37.7% / 28.7% / 25.2%, summing to 91.6%) — unlike PCA, where PC1 typically dominates, FA's rotated factors are not meant to be ranked by importance; varimax explicitly redistributes variance to make each factor equally interpretable rather than preserving a single dominant axis.
:::

---
## Exercises

1. **Rotation comparison**: Re-run FA with `rotation='promax'` (oblique rotation,
   allows correlated factors). Compare the loading heatmap with varimax. Are the
   factors correlated in the promax solution?

2. **4-factor solution**: Re-run FA with `n_factors=4`. Check the communalities —
   does the extra factor improve them substantially, or does it represent noise?

3. **Regression with factor scores**: Use the three factor scores as predictors in
   an OLS regression to predict `mass_loss`. Do the regression coefficients
   confirm the interpretation of the factors?